!! POT SER QUE FALLI SI L'ORDINADOR NO TÉ PROU MEMÒRIA RAM

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utils.tobii import (
    compute_regression_metrics,
    detect_regressions,
    extract_fixations,
    get_hits,
    process_all_texts,
    read_all_data,
    read_data,
    z_score,
)

pd.set_option("display.max_columns", None)

In [ ]:
PROJECT = "../data/sexisme_20"
general = (
    pd.read_csv(PROJECT + " general.tsv", sep="\t")
    .drop(["Event", "Recording timestamp", "Computer timestamp"], axis=1)
    .drop_duplicates()
)
general

In [ ]:
anotacions = pd.read_csv(PROJECT + " anotacions.tsv", sep="\t").drop(
    ["Recording timestamp", "Computer timestamp"], axis=1
)
anotacions = anotacions[anotacions["Event"].isin(["TextStart", "TextEnd", "KeyboardEvent"])]
records = []

for _, row in anotacions.iterrows():
    event = row["Event"]
    value = str(row["Event value"])

    if event == "TextStart":
        # Check if value is a plain number (text ID) or "Text (X)" format
        if not value.startswith("Text"):
            current_text_num = value
            keyboard_events = []  # Reset keyboard events for new text number

    elif event == "KeyboardEvent":
        keyboard_events.append(value)
        if len(keyboard_events) == 2:
            records.append(
                {
                    "user": row["Participant name"],
                    "text": current_text_num,
                    "sexist": keyboard_events[0],
                    "confidence": keyboard_events[1],
                }
            )
    elif len(keyboard_events) > 2:
        raise ValueError

In [ ]:
anotacions = pd.DataFrame(records, columns=["user", "text", "sexist", "confidence"])
anotacions.text = anotacions.text.map({"76)": "76"}).fillna(anotacions.text).astype(int)
assert anotacions.shape[0] == anotacions.user.nunique() * anotacions.text.nunique()
anotacions.sort_values(by=["user", "text"])

In [ ]:
# TODO: divide in participants and map recording -> participants
aoi_hit, calibration_dfs, participants, dfs = read_all_data("../data/all_parquets")

In [ ]:
import glob
import os

parquet_folder = "../data/all_parquets"
parquet_files = sorted(glob.glob(os.path.join(parquet_folder, "*.parquet")))

all_aoi_hit = []
all_calibration_dfs = []
all_participants = []
all_dfs = []

In [ ]:
text_dfs, aoi_cols_dict = process_all_texts(dfs, aoi_hit)

In [ ]:
# tsv_file = "sexism2 Recording3.tsv"
tsv_file = "sexism4 Data export"
aoi_hit, calibration_df, participants, df = read_data(tsv_file)

In [ ]:
text_dfs, aoi_cols_dict = process_all_texts(df, aoi_hit)

In [ ]:
text_df = text_dfs["2"]
aoi_cols = aoi_cols_dict["2"]

In [ ]:
fixations = extract_fixations(text_df)
fixations

In [ ]:
regressions = detect_regressions(fixations)
metrics = compute_regression_metrics(fixations, regressions)
regressions

# Other

In [ ]:
# això té sentit si deixem totes les columnes d'AOI
if aoi_hit[0] in text_df.columns:
    aoi_data = text_df[aoi_cols].fillna(0)

    # Check for simultaneous hits
    hits_per_row = aoi_data.sum(axis=1)
    multi_hits = hits_per_row[hits_per_row > 1]

    text_df.loc[multi_hits.index].drop(set(aoi_cols) - set(aoi_cols[152:154]), axis=1)

In [ ]:
# df[~df.Event.isna()]

In [ ]:
# raw_df[~raw_df.Event.isna()]

In [ ]:
# df[["gaze_x", "gaze_y"]]

In [ ]:
# df[df[aoi_hit[0]] != 0]

# Pupil diameter

In [ ]:
# hits to int
df = z_score(df, "Pupil diameter filtered", "participant")

In [ ]:
text_hits = get_hits(df, aoi_hit, "participant", normalize=True)

In [ ]:
# todo: is there a "text" column?, else -> create it
# -> unique [text, participant] to check if they did all the texts.

In [ ]:
texts = {t.split(" - ")[1].rsplit("]", 1)[0] for t in aoi_hit}
# texts = [t.split("[")[1].split()[0] for t in aoi_hit] # old?
text = list(texts)[2]
print(text)
plt.scatter(text_hits[text]["hits"], text_hits[text]["z_pupil"])
plt.show()